# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

In [2]:
%reload_ext dotenv

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [3]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "/Users/mperry/Desktop/deploying_AI_repo/deploying-ai/05_src/documents/managing_oneself.pdf"

loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

#PyPDFLoader loads one Document object per PDF page. For each, we can easily access:
#The string content of the page;
#Metadata containing the file name and page number.

13


In [4]:
# Join pages (13 document objects)

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

In [5]:
# Lets check
document_text[:500]

'www.hbr.org\nB\n \nEST  \n \nOF  HBR 1999\n \nManaging Oneself\n \nby Peter F . Drucker\n \n•\n \nIncluded with this full-text \n \nHarvard Business Review\n \n article:\nThe Idea in Brief—the core idea\nThe Idea in Practice—putting the idea to work\n \n1\n \nArticle Summary\n \n2\n \nManaging Oneself\nA list of related materials, with annotations to guide further\nexploration of the article’s ideas and applications\n \n12\n \nFurther Reading\nSuccess in the knowledge \neconomy comes to those who \nknow themselves—their \nstrengths'

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [6]:
from openai import OpenAI
from pydantic import BaseModel, Field
import os

client = OpenAI(
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1'
)

class Doc_Summary(BaseModel):
    Author: str
    Title: str
    Relevance: str = Field(description="a statement, no longer than one paragraph, that explains why this article is relevant for an AI professional in their professional development.")
    Summary: str = Field(description="a concise and succinct summary no longer than 1000 tokens.")
    Tone: str = Field(description="The tone used to generate the summary.")
    InputTokens: int = 0
    OutputTokens: int = 0

# Instructions stored separately
instructions = (
    "You are a helpful assistant for summarizing documents. "
    "You will be given the text of a document, and your task is to provide a concise summary "
    "along with some metadata about the document. "
    "Write the Summary in GenZ slang. Use terms like 'no cap', 'fr fr', 'lowkey', 'highkey', "
    "'bet', 'slay', 'it's giving...', 'the vibes', 'bussin', 'mid', 'rizz', 'ate', 'periodt', "
    "'say less', 'on god', 'hits different', 'main character energy', etc. "
    "Keep it casual, punchy, and unmistakably Gen Z. "
    "Set the Tone field to 'GenZ slang'."
)

# Context added dynamically via formatted string
user_prompt = f"Please summarize the following document:\n\n{document_text}"

response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt},
    ],
    text_format=Doc_Summary,
)

summary = response.output_parsed

# Populate token counts from response object, not from the model
summary.InputTokens = response.usage.input_tokens
summary.OutputTokens = response.usage.output_tokens


In [7]:
summary

Doc_Summary(Author='Peter F. Drucker', Title='Managing Oneself', Relevance='This article is a must-read for AI professionals who need to enhance their self-awareness and career adaptability in a rapidly changing industry, emphasizing the importance of personal strengths and values in professional success.', Summary="Aight, listen up! Drucker’s talk is all about being your own boss in the workplace, no cap. We’re living in a world where grinding hard can get you to the top, but you gotta know yourself first. It's giving main character energy—you gotta figure out your strengths, how you roll with others, and what vibes fit you best. Self-reflection is key—like, keep track of your wins and losses to know where you slay and where you need work. Also, find a work environment that’s your jam, or else you’ll just be vibing mid. Plus, recognize your values to avoid mismatched work drama, for real. Ultimately, managing yourself means stepping up like the CEO of your own life and making sure you

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [19]:
# Summarization metric
# The summarization metric uses LLM-as-a-judge to determine whether your LLM (application) is 
# generating factually correct summaries while including the necessary details from the original text. 
# In a summarization task within deepeval, the original text refers to the input while the summary is the actual_output.

from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric
from deepeval.models import GPTModel
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams
import os

# Build the model using your gateway
model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    _openai_api_key='any_value',  # placeholder — actual auth goes via x-api-key header
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

# Build the test case
test_case = LLMTestCase(
    input=user_prompt,
    actual_output=summary.Summary,
)

# Build the metric, passing the configured model
summarization_metric = SummarizationMetric(
    threshold=0.5,
    model=model,
    assessment_questions=[
        "Is the main topic or subject of the document clearly identified?",
        "Are the key findings, claims, or arguments presented?",
        "Are any specific examples, data points, or evidence mentioned?",
        "Are the authors or sources attributed?",
        "Is the conclusion or main takeaway stated?",
    ],
)

# Clarity / coherence check
coherence_metric = GEval(
    name="Coherence",
    evaluation_steps=[
        "Evaluate whether the response uses clear and direct language.",
        "Check if the explanation avoids jargon or explains it when used.",
        "Assess whether complex ideas are presented in a way that's easy to follow.",
        "Identify any vague or confusing parts that reduce understanding.",
        "Determine if the response is structured in a logical manner that enhances comprehension.",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,             
    threshold=0.7,         
)

genzified_tone = GEval( #Are we really doing the GenZ tone?
    name="GenZ Tone",
    evaluation_steps=[
        "Determine whether the actual output maintains a GenZ slang tone consistently throughout, rather than slipping into formal or neutral language.",
        "Evaluate if the language in the actual output reflects authentic GenZ expressions (e.g., 'no cap', 'fr', 'lowkey', 'slay', 'it's giving', 'bussin', 'mid', 'periodt', 'rizz', 'ate') used naturally rather than forced or sprinkled in awkwardly.",
        "Ensure the actual output stays casual, punchy, and unmistakably Gen Z, avoiding stiff, corporate, or overly formal phrasing.",
        "Check whether the sentence structure and rhythm feel conversational and internet-native, rather than reading like a traditional academic or professional summary rewritten with slang bolted on.",
        "Assess whether the actual output embraces informality and playfulness while still conveying the key information clearly and coherently.",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
    threshold=0.7,
)

pii_leakage = GEval( #Personal info leak?
    name="PII Leakage",
    evaluation_steps=[
        "Check whether the summary includes personal information (names, contact details, affiliations) that does NOT appear in the original 'Managing Oneself' text — such information would indicate hallucination.",
        "Verify that any names mentioned in the summary (e.g., Peter Drucker, historical figures referenced in the essay) are accurately attributed and appear in the source document, rather than being fabricated.",
        "Identify any hallucinated personal details — such as invented phone numbers, email addresses, home addresses, or biographical facts not present in the source — that could misrepresent real people.",
        "Ensure the summary does not introduce sensitive personal information about the author or referenced individuals beyond what is explicitly discussed in the original text.",
        "Confirm the summary avoids making up quotes, anecdotes, or personal claims attributed to Drucker or others that are not supported by the source document.",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.INPUT],
    model=model,
    threshold=0.9,
)
# Run all three on the same summary in one call
evaluate(
    test_cases=[test_case],
    metrics=[summarization_metric, coherence_metric, genzified_tone, pii_leakage],
)


✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Coherence [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest GenZ Tone [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest PII Leakage [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

Output()



Metrics Summary

  - ✅ Summarization (score: 0.625, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.62 because the summary includes extra information that is not present in the original text, which may lead to misinterpretation of the original message. However, there are no contradictions, which helps maintain some level of accuracy., error: None)
  - ❌ Coherence [GEval] (score: 0.36691929647115373, threshold: 0.7, strict: False, evaluation model: gpt-4o-mini, reason: The response uses clear language but relies heavily on informal slang, which may confuse some readers. While it presents the idea of self-management effectively, the use of phrases like 'grinding hard' and 'vibing mid' could alienate those unfamiliar with such terminology. Additionally, the structure lacks a logical flow, making it harder to follow the main points about self-reflection and work environment. Overall, while the core message is present, the execution diminishes clarity 

✓ Evaluation completed 🎉! (time taken: 10.99s | token cost: 0.006841199999999999 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=False, metrics_data=[MetricData(name='Summarization', threshold=0.5, success=True, score=0.625, reason='The score is 0.62 because the summary includes extra information that is not present in the original text, which may lead to misinterpretation of the original message. However, there are no contradictions, which helps maintain some level of accuracy.', strict_mode=False, evaluation_model='gpt-4o-mini', error=None, evaluation_cost=0.004592849999999999, verbose_logs='Truths (limit=None):\n[\n    "The document is an article titled \'Managing Oneself\' by Peter F. Drucker, published in the Harvard Business Review.",\n    "The article discusses the importance of self-management in the knowledge economy.",\n    "Drucker emphasizes that individuals must take responsibility for their own careers.",\n    "The article suggests that success in the knowledge economy comes to those who know their strengths, values, and how they

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [21]:
# Enhancement

# Rerun evaluate() but save the result this time so we can pull 
# the scores and reasons out of it.

first_eval = evaluate(
    test_cases=[test_case],
    metrics=[summarization_metric, coherence_metric, genzified_tone, pii_leakage],
)

# Pull out the score and reason for each metric from the first run
first_feedback = ""
for tr in first_eval.test_results:
    for m in tr.metrics_data:
        first_feedback += f"- {m.name} (score: {round(m.score, 3)}): {m.reason}\n"

print(first_feedback)

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Coherence [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest GenZ Tone [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest PII Leakage [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

Output()



Metrics Summary

  - ✅ Summarization (score: 0.5, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.50 because the summary includes several pieces of extra information that were not present in the original text, which may mislead the reader about the original content's focus and intent., error: None)
  - ❌ Coherence [GEval] (score: 0.37177821806035527, threshold: 0.7, strict: False, evaluation model: gpt-4o-mini, reason: The response uses informal language and slang, which may hinder clarity for some audiences. While it conveys the main ideas about self-management and self-reflection, the lack of structure and the use of jargon like 'vibing mid' and 'main character energy' could confuse readers. Additionally, complex ideas are not presented in a straightforward manner, making it harder to follow the key points., error: None)
  - ✅ GenZ Tone [GEval] (score: 0.9777299861174692, threshold: 0.7, strict: False, evaluation model: gpt-4o-mini, reason: The 

✓ Evaluation completed 🎉! (time taken: 26.1s | token cost: 0.0067746 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

- Summarization (score: 0.5): The score is 0.50 because the summary includes several pieces of extra information that were not present in the original text, which may mislead the reader about the original content's focus and intent.
- Coherence [GEval] (score: 0.372): The response uses informal language and slang, which may hinder clarity for some audiences. While it conveys the main ideas about self-management and self-reflection, the lack of structure and the use of jargon like 'vibing mid' and 'main character energy' could confuse readers. Additionally, complex ideas are not presented in a straightforward manner, making it harder to follow the key points.
- GenZ Tone [GEval] (score: 0.978): The response consistently maintains a GenZ slang tone throughout, using authentic expressions like 'no cap', 'it's giving', 'slay', and 'periodt' naturally. The language is casual and punchy, avoiding any formal phrasing, and the sentence structure feels conversational and internet-native. It emb

In [22]:
# Build a new prompt that includes the original doc, the previous summary, 
# and the evaluation feedback. 

refine_instructions = (
    "You are revising a previously generated document summary. "
    "You will be given the original document, the previous summary, and "
    "evaluation feedback with scores and reasons. "
    "Produce an improved summary that fixes the weaknesses called out in the feedback. "
    "Keep the GenZ slang tone, do not switch to formal writing. "
    "Use terms like 'no cap', 'fr fr', 'lowkey', 'highkey', 'bet', 'slay', "
    "'it's giving...', 'the vibes', 'bussin', 'mid', 'rizz', 'ate', 'periodt', etc. "
    "Make sure every claim is grounded in the original document, no made-up quotes or details. "
    "Keep the summary under 1000 tokens. "
    "Set the Tone field to 'GenZ slang'."
)

refine_user_prompt = f"""Please revise the summary below based on the evaluation feedback.

ORIGINAL DOCUMENT:
{document_text}

PREVIOUS SUMMARY:
{summary.Summary}

EVALUATION FEEDBACK:
{first_feedback}
"""

refined_response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "developer", "content": refine_instructions},
        {"role": "user", "content": refine_user_prompt},
    ],
    text_format=Doc_Summary,
)

refined_summary = refined_response.output_parsed

# Populate token counts from response object
refined_summary.InputTokens = refined_response.usage.input_tokens
refined_summary.OutputTokens = refined_response.usage.output_tokens

refined_summary

Doc_Summary(Author='Peter F. Drucker', Title='Managing Oneself', Relevance="Drucker’s concepts of self-management and self-awareness are crucial for AI professionals navigating their careers in a fast-paced, ever-evolving tech landscape. Understanding one's strengths, values, and working style not only improves individual performance but also enhances team dynamics, which is essential for successful collaboration in AI development.", Summary='Yo, let’s get into it! Drucker’s piece is all about taking charge of your career like you’re the CEO of your own life, fr fr. He’s serving up real talk about self-awareness in this knowledge economy. 🚀 If you wanna crush it, you gotta know your strengths, vibes, and how you roll with others. Self-reflection is a must—track your wins and flops to figure out where you slay and which areas need some work. Find your ideal work environment; if you’re in the wrong spot, it’s gonna feel mid. The values check is also key so you don’t end up in a workplace

In [23]:
# Now re-evaluate the refined summary using the same four metrics.
# Build a new test case with the refined summary as the actual_output.

refined_test_case = LLMTestCase(
    input=user_prompt,
    actual_output=refined_summary.Summary,
)

evaluate(
    test_cases=[refined_test_case],
    metrics=[summarization_metric, coherence_metric, genzified_tone, pii_leakage],
)

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Coherence [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest GenZ Tone [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest PII Leakage [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

Output()

...

ERROR:root:OpenAI Error: Request timed out. Retrying: 1 time(s)...

ERROR:root:OpenAI Error: Request timed out. Retrying: 1 time(s)...



Metrics Summary

  - ✅ Summarization (score: 0.5714285714285714, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.57 because the summary includes extra information that was not present in the original text, such as self-reflection, tracking successes and failures, finding an ideal work environment, a values check, and workplace drama. This additional content may mislead readers about the original message, affecting the overall quality of the summary., error: None)
  - ❌ Coherence [GEval] (score: 0.5069991993580631, threshold: 0.7, strict: False, evaluation model: gpt-4o-mini, reason: The response uses clear and direct language, making the main ideas accessible. However, it relies heavily on informal language and slang, which may confuse some readers and detracts from the clarity of the message. While it presents complex ideas about self-awareness and career management, the use of jargon like 'vibes' and 'slay' could alienate those unfamiliar with s

✓ Evaluation completed 🎉! (time taken: 46.29s | token cost: 0.006745649999999999 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=False, metrics_data=[MetricData(name='Summarization', threshold=0.5, success=True, score=0.5714285714285714, reason='The score is 0.57 because the summary includes extra information that was not present in the original text, such as self-reflection, tracking successes and failures, finding an ideal work environment, a values check, and workplace drama. This additional content may mislead readers about the original message, affecting the overall quality of the summary.', strict_mode=False, evaluation_model='gpt-4o-mini', error=None, evaluation_cost=0.004507499999999999, verbose_logs='Truths (limit=None):\n[\n    "The document is an article titled \'Managing Oneself\' by Peter F. Drucker, published in the Harvard Business Review.",\n    "The article discusses the importance of self-management in the knowledge economy.",\n    "Drucker emphasizes that individuals must take responsibility for their own careers.",\n    "Th

Please, do not forget to add your comments.

From observing the new evaluation scores, the refined summary is better but still the scores fall below the thresholds. This is mainly due to the language I decided to use for my summary - GenZ slang! The noted improvement from the enhancement steps tells me my enhancement is working though so I am happy with that! This is very cool.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
